# 2 — Federating k-means across cohorts

What crosses the boundary is **sufficient statistics**. No cohort ships a row.

The partition is found by **k-medoids**, not Lloyd's algorithm. Lloyd's averages
points into centroids, so it needs coordinates — and the pooled coordinates would
be every cohort's samples side by side, which is the one thing that must not leave
a project. The objective k-means minimises depends only on pairwise distances
(Huygens' theorem), and the pooled distance matrix is recoverable exactly from
sufficient statistics. So the partition is found centrally without the coordinates
existing anywhere.

This is the part of the k-means story that federates cleanly, and it holds
regardless of what you think about *p*-values on clusters.

In [ ]:
import os, shutil, subprocess, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Everything is written here, so nothing lands in the repository.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the
    command line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd if isinstance(cmd, list) else [cmd]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

In [ ]:
# The SLE data is not in the repository -- download it from Zenodo
# (doi:10.5281/zenodo.20342569) and point SLE at the unpacked folder.
SLE = Path(os.environ.get('SLE', '../data/SLE_doi.10.5281_zenodo_20342569'))
REAL = (SLE / 'abundance.csv').exists()

if REAL:
    NBOOT, TOPVAR = 1000, 100
    abundance = pd.read_csv(SLE / 'abundance.csv').set_index('SampleId')
    meta = pd.read_csv(SLE / 'sample-metadata.csv').set_index('SampleId')
    meta = meta[meta['Included_in_study'] == 'Included']
    abundance = abundance.loc[meta.index]
    FEATURE_MAP = str(SLE / 'feature_metadata.txt')
    print(f'SLE data: {abundance.shape[0]} samples x {abundance.shape[1]} reagents')
else:
    # A stand-in with the same shape of problem, so every command below runs
    # unchanged without the download: two batches, a case/control split, and a
    # few proteins measured by more than one reagent.
    NBOOT, TOPVAR = 40, 20
    rng = np.random.default_rng(0)
    n, p = 75, 30
    ids = [f'S{i:03d}' for i in range(n)]
    drivers = rng.normal(size=(n, 6))
    X = np.exp(rng.normal(3, 1, size=(1, p)) + drivers @ rng.normal(size=(6, p))
               + rng.normal(scale=0.3, size=(n, p)))
    seqs = [f'seq.{1000+j}.{j%7}' for j in range(p)]
    abundance = pd.DataFrame(X, index=pd.Index(ids, name='SampleId'), columns=seqs)
    batch = np.where(np.arange(n) % 3 == 0, 'B', 'A')
    abundance.loc[batch == 'B'] *= 1.6                     # a real batch shift
    meta = pd.DataFrame({
        'DonorId': ids, 'Batch': batch,
        'Group': np.where(rng.random(n) < 0.25, 'HV', 'SLE'),
        'Sex': rng.choice(['F', 'M'], n, p=[0.85, 0.15]),
        'Age_group': rng.choice(['26-30', '31-35', '36-40', '41-45'], n),
        'Disease_activity': rng.choice(['Remission', 'LDA', 'MDA', 'HDA'], n),
        'SLEDAI_2K': rng.integers(0, 14, n)}, index=pd.Index(ids, name='SampleId'))
    # names, with three proteins deliberately measured twice
    gene = [f'G{j:02d}' for j in range(p)]
    for a, b in [(1, 2), (10, 11), (20, 21)]:
        gene[b] = gene[a]
    fm = pd.DataFrame({'SeqId': seqs, 'TargetFullName': gene, 'GeneSymbol': gene})
    dup = fm['GeneSymbol'].duplicated(keep=False)
    fm.loc[dup, 'GeneSymbol'] = fm.loc[dup, 'GeneSymbol'] + '_' + fm.loc[dup, 'SeqId']
    fm.to_csv('feature_metadata.txt', sep='\t', index=False)
    FEATURE_MAP = 'feature_metadata.txt'
    print('SLE data not found -- using a stand-in of the same shape.')
    print(f'stand-in: {abundance.shape[0]} samples x {abundance.shape[1]} reagents')

In [ ]:
# Three cohorts, donors kept whole so repeat timepoints never straddle a boundary.
rng = np.random.default_rng(42)
donors = meta.groupby('DonorId').size().index.to_numpy()
who = dict(zip(rng.permutation(donors), range(len(donors))))
which = meta['DonorId'].map(lambda d: 'ABC'[who[d] % 3])

# SLEDAI banded, so it reads as a strip rather than fifteen shades of one colour.
out = meta.copy()
out['SLEDAI_band'] = pd.cut(pd.to_numeric(out['SLEDAI_2K'], errors='coerce'),
                            [-0.1, 0, 4, 8, 30], labels=['0', '1-4', '5-8', '9+'])
out = out.astype({'SLEDAI_band': str}).replace('nan', 'NA').fillna('NA')
out.to_csv('meta.csv')
abundance.to_csv('cohort_all.csv')
for c in 'ABC':
    abundance.loc[which[which == c].index].to_csv(f'cohort{c}.csv')
print({c: int((which == c).sum()) for c in 'ABC'})

In [ ]:
run(['pvclust-py', 'project-features', '--project', 'all',
     '--matrix', 'cohort_all.csv', '--log2',
     '--adjust', 'combat', '--batch-col', 'Batch', '--protect', 'Group',
     '--metadata', 'meta.csv', '--top-variable', str(TOPVAR),
     '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol'])

In [ ]:
# The flags every command shares. Written out in full each time below, so you can
# copy any single cell straight into a terminal.
COMMON = ['--log2', '--adjust', 'combat', '--batch-col', 'Batch',
          '--protect', 'Group', '--metadata', 'meta.csv',
          '--shared-features', 'all_features.csv',
          '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol']
DIST = ['--dist', 'correlation', '--linkage', 'average']
print(' '.join(COMMON))

## What leaves each cohort

Four `p x p` matrices: pairwise counts, sums, sums of squares, and the Gram matrix.
Every entry is a sum over rows, which is why they add across cohorts. This step is
`pvclust-py`'s — it does not depend on how you intend to cluster.

**The privacy rule is `p < n`.** The Gram matrix has rank `min(n, p)`, so a cohort
with fewer samples than objects exposes its row space completely. At `n = 1` the
Gram is rank one and returns the row itself (Homer et al., 2008).

In [ ]:
for c in 'ABC':
    run(['pvclust-py', 'project-stats', '--project', f'cohort{c}',
         '--matrix', f'cohort{c}.csv', *COMMON])

## The pooled partition

In [ ]:
STATS = [a for c in 'ABC' for a in ('--stats', f'cohort{c}_stats.npz')]
run(['kmeans-py', 'aggregate', '--labels', 'cohortA_labels.txt', '--k', '10',
     *STATS, '--dist', 'correlation', '--linkage', 'average'])

In [ ]:
part = pd.read_csv('federated_kmeans_partition.csv')
print(part[['n_members']].T.to_string())
for _, row in part.iterrows():
    print(f"  n={row['n_members']:3d}  {row['members'][:72]}")

## Does the pooled partition hold in each cohort?

The honest local check, and it needs no *p*-value. Cluster each cohort on its own,
then measure how far its partition agrees with the pooled one. Adjusted Rand is 1
for identical partitions and about 0 for unrelated ones, and it already corrects
for agreement expected by chance.

In [ ]:
from sklearn.metrics import adjusted_rand_score

pooled = {}
for i, row in part.iterrows():
    for m in row['members'].split(';'):
        pooled[m] = i

for c in 'ABC':
    run(['kmeans-py', 'cluster', '--project', f'local{c}', '--k', '10',
         '--matrix', f'cohort{c}.csv', '--cluster', 'columns', *COMMON])
    a = pd.read_csv(f'local{c}_assignment.csv', index_col=0)['cluster']
    shared = [o for o in a.index if o in pooled]
    ari = adjusted_rand_score([pooled[o] for o in shared], a.loc[shared])
    print(f'cohort{c}: adjusted Rand vs the pooled partition = {ari:.3f}')

---
**How to read that number.** High agreement means the pooled partition is not an
artefact of pooling — each cohort would have found something close on its own. Low
agreement means the cohorts disagree about where the boundaries fall, which is
worth knowing before anyone builds on the pooled result.

It is a measure of reproducibility across cohorts, not a *p*-value, and it should
not be reported as one.